# 91) Scoring do universo da Trilha B com B2 refit

## Objetivo do notebook
Este notebook cria os artefatos operacionais necessarios para a analise substantiva da Trilha B.
O foco e refitar a especificacao vencedora **B2** na **base analitica completa da Trilha B**,
gerar scores para todo o universo e, so depois, derivar os recortes analiticos no subconjunto
`S = 0`.

## Granularidade e base de entrada
- A base de entrada e `df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet`.
- Cada linha representa um agricultor, isto e, a granularidade e de **nivel pessoa** (`cpf_agricultor`).
- Algumas covariaveis replicam atributos de familia por join em `id_familia`, como ja documentado na trilha analitica.

## Interpretacao correta do score
- O score oficial de priorizacao sera chamado de `score_pu_b2_raw`.
- Esse score e um **score operacional de PU Learning sob hipotese SCAR**, nao uma probabilidade calibrada literal de `Y = 1`.
- O campo `score_pu_b2_clipped` sera persistido apenas como apoio descritivo e visual.

## Regra substantiva de uso
O scoring e feito na **base analitica completa da Trilha B**. A analise substantiva de priorizacao,
no entanto, sera realizada somente no subconjunto `S = 0`, filtrado **apos** o scoring do universo.

In [ ]:
from __future__ import annotations

import json
import math
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pulearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost nao esta disponivel no ambiente. O notebook 91 exige a especificacao B2 sem fallback."
    ) from exc

SMOKE = False #True
SMOKE_MAX_ROWS = 300_000
SAVE_ANALYSIS_READY = True

SEED = 42
TARGET_COL = "s"
GROUP_COL = "id_familia"
ID_COLS = ["cpf_agricultor", "id_familia"]
META_COLS = ["ibge"]
EXCLUDED_FROM_MODEL = [TARGET_COL] + ID_COLS + META_COLS

TOP_ABSOLUTE = 100_000
TOP_PERCENT = 0.01


def find_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz do projeto a partir do diretorio atual.")


def memory_mb(df: pd.DataFrame) -> float:
    return float(df.memory_usage(deep=True).sum() / 1024**2)


def relpath_str(path: Path, root: Path) -> str:
    try:
        return str(path.resolve().relative_to(root.resolve())).replace(chr(92), "/")
    except Exception:
        return str(path.resolve())


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/91_scoring_universo_trilhaB_b2_refit.ipynb"
DATA_PATH = PROJECT_ROOT / "dados/dados_processados/df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet"
OUTPUT_DIR = PROJECT_ROOT / "dados/dados_processados/outputs_pu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_MANIFEST_PATH = OUTPUT_DIR / "trilhaB_artifacts_manifest.parquet"
TEMPLATE_MODEL_PATH = OUTPUT_DIR / "trilhaB_best_model_pipeline.pkl"

artifact_suffix = "_smoke" if SMOKE else ""

SCORES_ALL_PATH = OUTPUT_DIR / f"trilhaB_b2_scores_all{artifact_suffix}.parquet"
SCORES_S0_PATH = OUTPUT_DIR / f"trilhaB_b2_scores_s0{artifact_suffix}.parquet"
RANKING_S0_FULL_PATH = OUTPUT_DIR / f"trilhaB_b2_ranking_s0_full{artifact_suffix}.parquet"
RANKING_S0_TOP100K_PATH = OUTPUT_DIR / f"trilhaB_b2_ranking_s0_top100k{artifact_suffix}.parquet"
RANKING_S0_BOTTOM100K_PATH = OUTPUT_DIR / f"trilhaB_b2_ranking_s0_bottom100k{artifact_suffix}.parquet"
RANKING_S0_TOP1PCT_PATH = OUTPUT_DIR / f"trilhaB_b2_ranking_s0_top1pct{artifact_suffix}.parquet"
RANKING_S0_BOTTOM1PCT_PATH = OUTPUT_DIR / f"trilhaB_b2_ranking_s0_bottom1pct{artifact_suffix}.parquet"
MANIFEST_PATH = OUTPUT_DIR / f"trilhaB_b2_scoring_manifest{artifact_suffix}.parquet"
REFIT_MODEL_PATH = OUTPUT_DIR / f"trilhaB_b2_model_refit_all{artifact_suffix}.pkl"
FEATURE_NAMES_PATH = OUTPUT_DIR / f"trilhaB_b2_feature_names_refit{artifact_suffix}.parquet"
ANALYSIS_READY_PATH = OUTPUT_DIR / f"trilhaB_b2_scores_all_analysis_ready{artifact_suffix}.parquet"

print(f"Projeto: {PROJECT_ROOT}")
print(f"SMOKE: {SMOKE} | SMOKE_MAX_ROWS: {SMOKE_MAX_ROWS:,}")
print(f"Base de entrada: {relpath_str(DATA_PATH, PROJECT_ROOT)}")
print(f"Diretorio de saida: {relpath_str(OUTPUT_DIR, PROJECT_ROOT)}")
print(f"Sufixo de artefatos: {artifact_suffix or '[sem sufixo]'}")

## Metodologia adotada

1. A especificacao do **B2** e congelada a partir da `21_trilha_B_pu_classico.ipynb`.
2. Nao ha nova competicao entre B1, B2 e B3 nesta etapa.
3. O **refit** e realizado na base analitica completa da Trilha B, usando `s` como rotulo observado.
4. O scoring cobre todo o universo antes de qualquer filtragem substantiva.
5. O ranking oficial e definido por ordenacao decrescente de `score_pu_b2_raw`, com desempate deterministico por `cpf_agricultor`.
6. Os recortes analiticos sao extraidos apenas depois disso, no subconjunto `S = 0`.

In [ ]:
def sample_smoke_by_group(
    df: pd.DataFrame,
    group_col: str,
    max_rows: int,
    random_state: int,
) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.copy()

    group_sizes = (
        df.groupby(group_col, dropna=False, observed=True)
        .size()
        .reset_index(name="n_rows")
        .sample(frac=1.0, random_state=random_state)
        .reset_index(drop=True)
    )
    group_sizes["cum_rows"] = group_sizes["n_rows"].cumsum()

    if group_sizes["cum_rows"].ge(max_rows).any():
        cutoff_idx = int(group_sizes["cum_rows"].ge(max_rows).idxmax())
    else:
        cutoff_idx = len(group_sizes) - 1

    selected_groups = set(group_sizes.loc[:cutoff_idx, group_col].astype("string"))
    sampled = df[df[group_col].astype("string").isin(selected_groups)].copy()
    return sampled.reset_index(drop=True)


def get_git_metadata(project_root: Path) -> dict[str, object]:
    def run_git(args: list[str]) -> str:
        try:
            result = subprocess.run(
                ["git", *args],
                cwd=project_root,
                check=True,
                capture_output=True,
                text=True,
                encoding="utf-8",
            )
            return result.stdout.strip()
        except Exception:
            return ""

    branch = run_git(["branch", "--show-current"])
    commit = run_git(["rev-parse", "HEAD"])
    dirty = bool(run_git(["status", "--short"]))
    return {"git_branch": branch, "git_commit": commit, "git_is_dirty": dirty}


def get_positive_class_index(pu_step, positive_label: int = 1) -> tuple[int, str, list[int]]:
    candidates: list[tuple[str, np.ndarray]] = []
    if hasattr(pu_step, "classes_"):
        candidates.append(("pu_step.classes_", np.asarray(pu_step.classes_)))
    if hasattr(pu_step, "estimator_") and hasattr(pu_step.estimator_, "classes_"):
        candidates.append(("pu_step.estimator_.classes_", np.asarray(pu_step.estimator_.classes_)))
    if hasattr(pu_step, "estimator") and hasattr(pu_step.estimator, "classes_"):
        candidates.append(("pu_step.estimator.classes_", np.asarray(pu_step.estimator.classes_)))

    for source, labels in candidates:
        for idx, value in enumerate(labels.tolist()):
            if value == positive_label:
                return int(idx), source, [int(x) for x in labels.tolist()]

    raise ValueError(
        "Nao foi possivel mapear a classe positiva (1) em classes_. "
        "Sem esse mapeamento, o score de predict_proba nao e confiavel."
    )


def predict_positive_score(model_pipeline: Pipeline, X_ref: pd.DataFrame, positive_label: int = 1) -> np.ndarray:
    pu_step = model_pipeline.named_steps["pu"]
    pos_idx, _, _ = get_positive_class_index(pu_step, positive_label=positive_label)
    proba = np.asarray(model_pipeline.predict_proba(X_ref))
    if proba.ndim != 2 or proba.shape[1] <= pos_idx:
        raise ValueError(f"Saida inesperada de predict_proba: shape={proba.shape}, pos_idx={pos_idx}")
    return proba[:, pos_idx]


def extract_c_est(pu_step) -> float:
    for attr in ("c", "c_", "c_est", "c_est_"):
        if hasattr(pu_step, attr):
            value = getattr(pu_step, attr)
            try:
                return float(value)
            except Exception:
                continue
    return float("nan")


def mask_cpf_value(value: object) -> str:
    if pd.isna(value):
        return "<NA>"
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    if len(digits) < 4:
        return "***"
    return f"***{digits[-4:]}"


def mask_cpf_frame(df: pd.DataFrame, cpf_col: str = "cpf_agricultor") -> pd.DataFrame:
    masked = df.copy()
    if cpf_col in masked.columns:
        masked[cpf_col] = masked[cpf_col].map(mask_cpf_value)
    return masked

## Etapa 1 — Congelamento da especificação B2

Esta secao recupera a configuracao vencedora da Trilha B e documenta o que foi congelado para o refit.
O objetivo e evitar qualquer ambiguidade sobre qual modelo esta sendo reaplicado no universo completo.

In [ ]:
template_manifest = pd.read_parquet(TEMPLATE_MANIFEST_PATH)
if len(template_manifest) != 1:
    raise ValueError(
        f"O manifest de referencia deve ter exatamente uma linha. Linhas encontradas: {len(template_manifest)}"
    )

manifest_row = template_manifest.iloc[0].to_dict()
if manifest_row.get("best_raw_model_code") != "B2":
    raise ValueError(
        "O manifest de referencia nao aponta B2 como melhor modelo bruto. "
        f"Valor encontrado: {manifest_row.get('best_raw_model_code')}"
    )
if manifest_row.get("b2_base_estimator") != "XGBoostClassifier":
    raise ValueError(
        "A especificacao congelada do B2 exige XGBoostClassifier. "
        f"Valor encontrado no manifest: {manifest_row.get('b2_base_estimator')}"
    )

template_pipeline = joblib.load(TEMPLATE_MODEL_PATH)
template_feature_cols = list(template_pipeline.named_steps["preprocess"].feature_names_in_)

B2_HOLD_OUT_RATIO = float(manifest_row["hold_out_ratio"])
B2_XGB_PARAMS = {
    "n_estimators": 320,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "min_child_weight": 20,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": 0,
}

freeze_summary = pd.DataFrame(
    [
        {"item": "notebook_origem", "valor": manifest_row.get("notebook_origem")},
        {"item": "best_raw_model_code", "valor": manifest_row.get("best_raw_model_code")},
        {"item": "best_raw_model_name", "valor": manifest_row.get("best_raw_model_name")},
        {"item": "base_estimator", "valor": manifest_row.get("b2_base_estimator")},
        {"item": "hold_out_ratio", "valor": B2_HOLD_OUT_RATIO},
        {"item": "seed", "valor": SEED},
        {"item": "n_features", "valor": len(template_feature_cols)},
        {"item": "score_oficial", "valor": "score_pu_b2_raw"},
        {"item": "score_auxiliar", "valor": "score_pu_b2_clipped"},
    ]
)

print("B2 congelado com base na Trilha B; nao ha selecao de modelo nesta etapa.")
display(freeze_summary)

print("\nPrimeiras features da especificacao congelada:")
display(pd.DataFrame({"feature_name": template_feature_cols[:10]}))

## Etapa 2 — Carga da base analítica completa

Aqui carregamos a base consolidada da Trilha B e validamos as chaves minimas para o scoring.
Em `SMOKE`, a base e reduzida por amostragem de grupos (`id_familia`) sem alterar a logica do pipeline.

In [ ]:
df = pd.read_parquet(DATA_PATH)
rows_original_base = int(len(df))

if SMOKE:
    df = sample_smoke_by_group(df, group_col=GROUP_COL, max_rows=SMOKE_MAX_ROWS, random_state=SEED)

rows_execucao = int(len(df))

for col in ["cpf_agricultor", "id_familia", "ibge"]:
    if col not in df.columns:
        raise KeyError(f"Coluna obrigatoria ausente na base: {col}")
    df[col] = df[col].astype("string")

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
if df[TARGET_COL].isna().any():
    raise ValueError("A coluna s contem valores nao numericos ou missing apos coercao.")
if not set(df[TARGET_COL].astype(int).unique().tolist()).issubset({0, 1}):
    raise ValueError("A coluna s precisa conter apenas 0 e 1.")
df[TARGET_COL] = df[TARGET_COL].astype("int8")
base_original_columns = list(df.columns)

key_missing = pd.DataFrame(
    {
        "coluna": ["cpf_agricultor", "id_familia", "ibge"],
        "missing": [int(df[c].isna().sum()) for c in ["cpf_agricultor", "id_familia", "ibge"]],
    }
)
duplicate_cpf = int(df["cpf_agricultor"].duplicated().sum())
s_distribution = df[TARGET_COL].value_counts(dropna=False).sort_index().rename_axis("s").reset_index(name="n")

base_summary = pd.DataFrame(
    [
        {"metrica": "rows_original_base", "valor": rows_original_base},
        {"metrica": "rows_execucao", "valor": rows_execucao},
        {"metrica": "cols_execucao", "valor": int(df.shape[1])},
        {"metrica": "memoria_mb", "valor": round(memory_mb(df), 2)},
        {"metrica": "duplicated_cpf_agricultor", "valor": duplicate_cpf},
    ]
)

print("Resumo da base de execucao:")
display(base_summary)
print("\nMissing em chaves principais:")
display(key_missing)
print("\nDistribuicao de s:")
display(s_distribution)

## Etapas 3 a 5 — Reconstrução das features, refit do B2 e scoring do universo

Este bloco recria o conjunto exato de features do B2, refita o modelo na base analitica completa
da Trilha B e gera os scores oficiais do universo inteiro. A interpretacao continua sendo a mesma
da trilha original: `score_pu_b2_raw` e o score operacional PU e `score_pu_b2_clipped` e apenas auxiliar.

In [ ]:
# ---------------------------------------------------------
# Reconstrucao fiel das features da especificacao B2
# ---------------------------------------------------------
#
# Objetivo:
# Garantir que o refit do universo use exatamente o mesmo conjunto
# de features elegiveis e o mesmo preprocessamento da Trilha B.
#
# Interpretacao:
# Qualquer divergencia aqui compromete a comparabilidade com o B2
# consolidado no notebook 21.

df["st_missing_lat_imovel_principal"] = df["lat_imovel_principal"].isna().astype("int8")
df["st_missing_lon_imovel_principal"] = df["lon_imovel_principal"].isna().astype("int8")

feature_cols_model = [c for c in df.columns if c not in EXCLUDED_FROM_MODEL]
missing_features = [c for c in template_feature_cols if c not in feature_cols_model]
extra_features = [c for c in feature_cols_model if c not in template_feature_cols]

if missing_features or extra_features:
    raise ValueError(
        "A composicao de features divergiu da especificacao congelada do B2. "
        f"missing_features={missing_features}; extra_features={extra_features}"
    )

X_all = df[template_feature_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
y_all = df[TARGET_COL].astype("int8").to_numpy()

numeric_feature_cols = list(X_all.columns)
feature_names_refit = pd.DataFrame(
    {
        "feature_order": np.arange(len(numeric_feature_cols), dtype="int32"),
        "feature_name": numeric_feature_cols,
        "dtype_original": [str(df[c].dtype) for c in numeric_feature_cols],
    }
)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
preprocess_refit = ColumnTransformer(
    transformers=[("numeric", numeric_pipeline, numeric_feature_cols)],
    remainder="drop",
    verbose_feature_names_out=False,
)

estimator_b2 = XGBClassifier(**B2_XGB_PARAMS)
pu_b2 = pulearn.ElkanotoPuClassifier(
    estimator=estimator_b2,
    hold_out_ratio=B2_HOLD_OUT_RATIO,
    random_state=SEED,
)
model_b2_refit = Pipeline(
    steps=[
        ("preprocess", preprocess_refit),
        ("pu", pu_b2),
    ]
)

print("Iniciando refit do B2 na base analitica completa de execucao...")
fit_start = time.perf_counter()
model_b2_refit.fit(X_all, y_all)
fit_seconds = float(time.perf_counter() - fit_start)

pu_step_refit = model_b2_refit.named_steps["pu"]
proba_class_index_for_1, class_source_refit, class_labels_refit = get_positive_class_index(
    pu_step_refit,
    positive_label=1,
)
score_pu_b2_raw = predict_positive_score(model_b2_refit, X_all, positive_label=1).astype("float64")

finite_scores = score_pu_b2_raw[np.isfinite(score_pu_b2_raw)]
if finite_scores.size == 0:
    raise ValueError("O scoring do universo retornou apenas valores nao finitos.")
score_pu_b2_raw = np.nan_to_num(
    score_pu_b2_raw,
    nan=0.0,
    posinf=float(finite_scores.max()),
    neginf=float(finite_scores.min()),
)
score_pu_b2_clipped = np.clip(score_pu_b2_raw, 0.0, 1.0).astype("float32")

c_est_refit = extract_c_est(pu_step_refit)
converged_refit = True

refit_summary = pd.DataFrame(
    [
        {"metrica": "fit_seconds", "valor": round(fit_seconds, 3)},
        {"metrica": "n_features", "valor": len(numeric_feature_cols)},
        {"metrica": "c_est_refit", "valor": c_est_refit},
        {"metrica": "class_source_refit", "valor": class_source_refit},
        {"metrica": "proba_class_index_for_1", "valor": proba_class_index_for_1},
        {"metrica": "rows_fit", "valor": len(X_all)},
    ]
)
display(refit_summary)

## Etapas 6 a 10 - Persistencia do universo scored, derivacao de `S = 0` e construcao do `analysis_ready`

Nesta etapa o score do universo e transformado em artefatos operacionais. Primeiro persistimos
a base slim do universo completo. Depois filtramos `S = 0`, ordenamos o ranking oficial e
geramos os recortes absolutos e relativos usados na analise substantiva.

Por fim, o notebook monta `trilhaB_b2_scores_all_analysis_ready.parquet` como um `left join`
da base analitica original com o resultado do artefato `scores_s0`, preservando as 47 colunas
de entrada e anexando apenas as colunas de score, ranking e flags da priorizacao em `S = 0`.

In [ ]:
# ---------------------------------------------------------
# Conversao do scoring em artefatos operacionais
# ---------------------------------------------------------
#
# O ranking oficial usa score_pu_b2_raw em ordem decrescente.
# Empates sao quebrados por cpf_agricultor em ordem crescente,
# replicando a logica deterministica adotada na Trilha B.

df_scores_all = df[["cpf_agricultor", "id_familia", "ibge", TARGET_COL]].copy()
df_scores_all = df_scores_all.rename(columns={TARGET_COL: "s"})
df_scores_all["score_pu_b2_raw"] = score_pu_b2_raw.astype("float64")
df_scores_all["score_pu_b2_clipped"] = score_pu_b2_clipped

df_scores_all = (
    df_scores_all
    .sort_values(["score_pu_b2_raw", "cpf_agricultor"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)
n_total = int(len(df_scores_all))
df_scores_all["ranking_global_raw"] = np.arange(1, n_total + 1, dtype="int32")
df_scores_all["percentil_score_raw"] = (
    (n_total - df_scores_all["ranking_global_raw"] + 1) / n_total
).astype("float32")

df_scores_s0 = (
    df_scores_all.loc[df_scores_all["s"] == 0].copy()
    .sort_values(["score_pu_b2_raw", "cpf_agricultor"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)
n_s0 = int(len(df_scores_s0))
df_scores_s0["ranking_s0_raw"] = np.arange(1, n_s0 + 1, dtype="int32")
if n_s0 > 0:
    df_scores_s0["percentil_s0_raw"] = (
        (n_s0 - df_scores_s0["ranking_s0_raw"] + 1) / n_s0
    ).astype("float32")
else:
    df_scores_s0["percentil_s0_raw"] = pd.Series(dtype="float32")

n_top100k = min(TOP_ABSOLUTE, n_s0)
n_bottom100k = min(TOP_ABSOLUTE, n_s0)
n_top1pct = max(int(math.ceil(TOP_PERCENT * n_s0)), 1) if n_s0 > 0 else 0
n_bottom1pct = max(int(math.ceil(TOP_PERCENT * n_s0)), 1) if n_s0 > 0 else 0

df_scores_s0["flag_top100k_raw"] = (df_scores_s0["ranking_s0_raw"] <= n_top100k).astype("int8")
df_scores_s0["flag_bottom100k_raw"] = (
    df_scores_s0["ranking_s0_raw"] > max(n_s0 - n_bottom100k, 0)
).astype("int8")
df_scores_s0["flag_top1pct_raw"] = (df_scores_s0["ranking_s0_raw"] <= n_top1pct).astype("int8")
df_scores_s0["flag_bottom1pct_raw"] = (
    df_scores_s0["ranking_s0_raw"] > max(n_s0 - n_bottom1pct, 0)
).astype("int8")

df_ranking_s0_full = df_scores_s0.copy()
df_ranking_s0_top100k = df_ranking_s0_full.loc[df_ranking_s0_full["flag_top100k_raw"] == 1].copy()
df_ranking_s0_bottom100k = df_ranking_s0_full.loc[df_ranking_s0_full["flag_bottom100k_raw"] == 1].copy()
df_ranking_s0_top1pct = df_ranking_s0_full.loc[df_ranking_s0_full["flag_top1pct_raw"] == 1].copy()
df_ranking_s0_bottom1pct = df_ranking_s0_full.loc[df_ranking_s0_full["flag_bottom1pct_raw"] == 1].copy()

git_metadata = get_git_metadata(PROJECT_ROOT)
pct_score_gt_1 = float((df_scores_all["score_pu_b2_raw"] > 1).mean()) if n_total > 0 else float("nan")

analysis_ready_score_cols = [
    "score_pu_b2_raw",
    "score_pu_b2_clipped",
    "ranking_global_raw",
    "percentil_score_raw",
    "ranking_s0_raw",
    "percentil_s0_raw",
    "flag_top100k_raw",
    "flag_bottom100k_raw",
    "flag_top1pct_raw",
    "flag_bottom1pct_raw",
]
analysis_ready_join_cols = ["cpf_agricultor", "id_familia", "ibge", "s"] + analysis_ready_score_cols
analysis_ready_rows = pd.NA
analysis_ready_cols = pd.NA
analysis_ready_score_nonnull = pd.NA

joblib.dump(model_b2_refit, REFIT_MODEL_PATH)
feature_names_refit.to_parquet(FEATURE_NAMES_PATH, index=False)
df_scores_all.to_parquet(SCORES_ALL_PATH, index=False)
df_scores_s0.to_parquet(SCORES_S0_PATH, index=False)
df_ranking_s0_full.to_parquet(RANKING_S0_FULL_PATH, index=False)
df_ranking_s0_top100k.to_parquet(RANKING_S0_TOP100K_PATH, index=False)
df_ranking_s0_bottom100k.to_parquet(RANKING_S0_BOTTOM100K_PATH, index=False)
df_ranking_s0_top1pct.to_parquet(RANKING_S0_TOP1PCT_PATH, index=False)
df_ranking_s0_bottom1pct.to_parquet(RANKING_S0_BOTTOM1PCT_PATH, index=False)

if SAVE_ANALYSIS_READY:
    # O analysis_ready preserva a base analitica original inteira e anexa
    # apenas o resultado do artefato scores_s0. Assim, qualquer analise
    # posterior pode contar o universo total e, ao mesmo tempo, identificar
    # quais agricultores s=0 receberam score, rank e flags substantivas.
    df_scores_s0_join = pd.read_parquet(SCORES_S0_PATH, columns=analysis_ready_join_cols)
    df_base_analysis_ready = df[base_original_columns].copy()
    df_analysis_ready = df_base_analysis_ready.merge(
        df_scores_s0_join,
        on=["cpf_agricultor", "id_familia", "ibge", "s"],
        how="left",
        validate="1:1",
    )

    for col in ["flag_top100k_raw", "flag_bottom100k_raw", "flag_top1pct_raw", "flag_bottom1pct_raw"]:
        df_analysis_ready[col] = df_analysis_ready[col].fillna(0).astype("int8")

    df_analysis_ready["ranking_global_raw"] = df_analysis_ready["ranking_global_raw"].astype("Int32")
    df_analysis_ready["ranking_s0_raw"] = df_analysis_ready["ranking_s0_raw"].astype("Int32")
    df_analysis_ready["percentil_score_raw"] = df_analysis_ready["percentil_score_raw"].astype("float32")
    df_analysis_ready["percentil_s0_raw"] = df_analysis_ready["percentil_s0_raw"].astype("float32")

    ordered_columns = base_original_columns + analysis_ready_score_cols
    df_analysis_ready = df_analysis_ready[ordered_columns]
    df_analysis_ready.to_parquet(ANALYSIS_READY_PATH, index=False)

    analysis_ready_rows = int(len(df_analysis_ready))
    analysis_ready_cols = int(df_analysis_ready.shape[1])
    analysis_ready_score_nonnull = int(df_analysis_ready["score_pu_b2_raw"].notna().sum())
else:
    df_analysis_ready = None

df_scoring_manifest = pd.DataFrame(
    [
        {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "notebook_origem_especificacao": manifest_row.get("notebook_origem"),
            "notebook_scoring": relpath_str(NOTEBOOK_PATH, PROJECT_ROOT),
            "data_path": relpath_str(DATA_PATH, PROJECT_ROOT),
            "output_dir": relpath_str(OUTPUT_DIR, PROJECT_ROOT),
            "smoke": bool(SMOKE),
            "smoke_max_rows": int(SMOKE_MAX_ROWS),
            "rows_original_base": rows_original_base,
            "rows_execucao": rows_execucao,
            "rows_s0": n_s0,
            "seed": SEED,
            "group_col": GROUP_COL,
            "hold_out_ratio": B2_HOLD_OUT_RATIO,
            "best_raw_model_code": manifest_row.get("best_raw_model_code"),
            "best_raw_model_name": manifest_row.get("best_raw_model_name"),
            "b2_base_estimator": manifest_row.get("b2_base_estimator"),
            "score_oficial": "score_pu_b2_raw",
            "score_auxiliar": "score_pu_b2_clipped",
            "ranking_oficial": "sort(score_pu_b2_raw desc, cpf_agricultor asc)",
            "top_bottom_absolute_rule": "100000 observacoes",
            "top_bottom_percent_rule": "ceil(0.01 * N_s0)",
            "pct_score_pu_b2_raw_gt_1": pct_score_gt_1,
            "fit_seconds": fit_seconds,
            "converged_refit": converged_refit,
            "c_est_refit": c_est_refit,
            "class_source_refit": class_source_refit,
            "class_labels_refit": json.dumps(class_labels_refit, ensure_ascii=False),
            "proba_class_index_for_1": proba_class_index_for_1,
            "xgb_params_json": json.dumps(B2_XGB_PARAMS, ensure_ascii=False, sort_keys=True),
            "feature_list_json": json.dumps(template_feature_cols, ensure_ascii=False),
            "feature_count": len(template_feature_cols),
            "analysis_ready_all_file": relpath_str(ANALYSIS_READY_PATH, PROJECT_ROOT) if SAVE_ANALYSIS_READY else pd.NA,
            "analysis_ready_join_source": relpath_str(SCORES_S0_PATH, PROJECT_ROOT) if SAVE_ANALYSIS_READY else pd.NA,
            "analysis_ready_rows": analysis_ready_rows,
            "analysis_ready_cols": analysis_ready_cols,
            "analysis_ready_score_nonnull": analysis_ready_score_nonnull,
            "metodologia_scar": (
                "score_pu_b2_raw e score operacional PU sob hipotese SCAR; "
                "nao e probabilidade calibrada literal de Y=1"
            ),
            **git_metadata,
        }
    ]
)
df_scoring_manifest.to_parquet(MANIFEST_PATH, index=False)

saved_artifacts_list = [
    relpath_str(SCORES_ALL_PATH, PROJECT_ROOT),
    relpath_str(SCORES_S0_PATH, PROJECT_ROOT),
    relpath_str(RANKING_S0_FULL_PATH, PROJECT_ROOT),
    relpath_str(RANKING_S0_TOP100K_PATH, PROJECT_ROOT),
    relpath_str(RANKING_S0_BOTTOM100K_PATH, PROJECT_ROOT),
    relpath_str(RANKING_S0_TOP1PCT_PATH, PROJECT_ROOT),
    relpath_str(RANKING_S0_BOTTOM1PCT_PATH, PROJECT_ROOT),
    relpath_str(REFIT_MODEL_PATH, PROJECT_ROOT),
    relpath_str(FEATURE_NAMES_PATH, PROJECT_ROOT),
    relpath_str(MANIFEST_PATH, PROJECT_ROOT),
]
if SAVE_ANALYSIS_READY:
    saved_artifacts_list.append(relpath_str(ANALYSIS_READY_PATH, PROJECT_ROOT))

saved_artifacts = pd.DataFrame({"artefato": saved_artifacts_list})
print("Artefatos persistidos nesta execucao:")
display(saved_artifacts)

## Etapas 11 a 13 — Diagnósticos, auditoria e leitura dos resultados

Os diagnosticos abaixo servem para verificar se o scoring do universo produziu um artefato coerente,
auditavel e utilizavel no artigo. Eles nao substituem uma avaliacao `train/test`, que ja foi feita no
notebook 21; aqui o objetivo e controlar a qualidade do refit e do ranking final do universo.

In [ ]:
print("Shapes dos principais artefatos:")
artifact_shape_rows = [
    {"artefato": "scores_all", "rows": len(df_scores_all), "cols": df_scores_all.shape[1]},
    {"artefato": "scores_s0", "rows": len(df_scores_s0), "cols": df_scores_s0.shape[1]},
    {"artefato": "ranking_s0_top100k", "rows": len(df_ranking_s0_top100k), "cols": df_ranking_s0_top100k.shape[1]},
    {"artefato": "ranking_s0_bottom100k", "rows": len(df_ranking_s0_bottom100k), "cols": df_ranking_s0_bottom100k.shape[1]},
    {"artefato": "ranking_s0_top1pct", "rows": len(df_ranking_s0_top1pct), "cols": df_ranking_s0_top1pct.shape[1]},
    {"artefato": "ranking_s0_bottom1pct", "rows": len(df_ranking_s0_bottom1pct), "cols": df_ranking_s0_bottom1pct.shape[1]},
]
if SAVE_ANALYSIS_READY and df_analysis_ready is not None:
    artifact_shape_rows.append({"artefato": "analysis_ready_all", "rows": len(df_analysis_ready), "cols": df_analysis_ready.shape[1]})
display(pd.DataFrame(artifact_shape_rows))

print("\nMissing em chaves da base scored:")
display(
    pd.DataFrame(
        {
            "coluna": ["cpf_agricultor", "id_familia", "ibge"],
            "missing_scores_all": [int(df_scores_all[c].isna().sum()) for c in ["cpf_agricultor", "id_familia", "ibge"]],
            "missing_scores_s0": [int(df_scores_s0[c].isna().sum()) for c in ["cpf_agricultor", "id_familia", "ibge"]],
        }
    )
)

print("\nDistribuicao de s na base scored:")
display(df_scores_all["s"].value_counts(dropna=False).sort_index().rename_axis("s").reset_index(name="n"))

score_percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
print("\nEstatisticas descritivas de score_pu_b2_raw:")
display(df_scores_all["score_pu_b2_raw"].describe(percentiles=score_percentiles).to_frame(name="score_pu_b2_raw"))

print("\nEstatisticas descritivas de score_pu_b2_clipped:")
display(df_scores_all["score_pu_b2_clipped"].describe(percentiles=score_percentiles).to_frame(name="score_pu_b2_clipped"))

pct_gt1 = float((df_scores_all["score_pu_b2_raw"] > 1).mean()) if len(df_scores_all) > 0 else float("nan")
print(f"\nPercentual de score_pu_b2_raw > 1: {pct_gt1:.4%}")

print("\nSeparacao do score por s (raw):")
display(df_scores_all.groupby("s")["score_pu_b2_raw"].describe(percentiles=score_percentiles))

print("\nSeparacao do score por s (clipped):")
display(df_scores_all.groupby("s")["score_pu_b2_clipped"].describe(percentiles=score_percentiles))

print("\nTamanho efetivo dos recortes substantivos em S=0:")
display(
    pd.DataFrame(
        [
            {"recorte": "top100k", "n": n_top100k},
            {"recorte": "bottom100k", "n": n_bottom100k},
            {"recorte": "top1pct", "n": n_top1pct},
            {"recorte": "bottom1pct", "n": n_bottom1pct},
        ]
    )
)

if SAVE_ANALYSIS_READY and df_analysis_ready is not None:
    print("\nDiagnosticos do analysis_ready (base original + scores_s0):")
    display(
        pd.DataFrame(
            [
                {"metrica": "rows_analysis_ready", "valor": int(len(df_analysis_ready))},
                {"metrica": "cols_analysis_ready", "valor": int(df_analysis_ready.shape[1])},
                {"metrica": "cpf_unicos", "valor": int(df_analysis_ready["cpf_agricultor"].nunique(dropna=False))},
                {"metrica": "score_preenchido_total", "valor": int(df_analysis_ready["score_pu_b2_raw"].notna().sum())},
                {"metrica": "score_preenchido_em_s0", "valor": int(df_analysis_ready.loc[df_analysis_ready["s"] == 0, "score_pu_b2_raw"].notna().sum())},
                {"metrica": "score_preenchido_em_s1", "valor": int(df_analysis_ready.loc[df_analysis_ready["s"] == 1, "score_pu_b2_raw"].notna().sum())},
            ]
        )
    )

    print("\nPreenchimento de score por s no analysis_ready:")
    display(
        df_analysis_ready.assign(score_preenchido=df_analysis_ready["score_pu_b2_raw"].notna())
        .groupby(["s", "score_preenchido"], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )

    print("\nInfo final do analysis_ready:")
    df_analysis_ready.info(show_counts=True)

top_cols = [
    "cpf_agricultor",
    "id_familia",
    "ibge",
    "s",
    "score_pu_b2_raw",
    "score_pu_b2_clipped",
    "ranking_s0_raw",
    "percentil_s0_raw",
    "flag_top100k_raw",
    "flag_top1pct_raw",
]
bottom_cols = [
    "cpf_agricultor",
    "id_familia",
    "ibge",
    "s",
    "score_pu_b2_raw",
    "score_pu_b2_clipped",
    "ranking_s0_raw",
    "percentil_s0_raw",
    "flag_bottom100k_raw",
    "flag_bottom1pct_raw",
]

print("\nTopo do ranking S=0 (CPF mascarado):")
display(mask_cpf_frame(df_ranking_s0_full[top_cols].head(10)))

print("\nBase do ranking S=0 (CPF mascarado):")
display(mask_cpf_frame(df_ranking_s0_full[bottom_cols].tail(10)))

print("\nInfo final do artefato slim do universo scored:")
df_scores_all.info(show_counts=True)

## Fechamento metodológico

- O **score oficial de priorizacao** deste notebook e `score_pu_b2_raw`.
- `score_pu_b2_raw` continua sendo um **score operacional PU sob SCAR**, nao uma probabilidade calibrada literal de `Y = 1`.
- `score_pu_b2_clipped` e mantido apenas como artefato auxiliar descritivo.
- A etapa presente nao reavalia B1, B2 e B3. Ela apenas **refita o B2 vencedor** para scoring do universo analitico.
- Uma extensao metodologica futura seria gerar `out-of-fold scoring` para reduzir vies in-sample na leitura substantiva do universo.